In [ ]:
import pandas as pd
import csv
import statistics
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
import os
import numpy as np
from pathlib import Path
from xmigcs import XMIGCS_ROOT_DIR

# 复制日志文件到本地
# scp ubuntu@10.11.218.152:/home/ubuntu/XMIGCS_DEPLOY/xmigcs/logs/navigate/20260422_155648_855669_commands.csv .

log_dir = f"{XMIGCS_ROOT_DIR}/logs/navigate"
# log_dir = f"{XMIGCS_ROOT_DIR}/logs/stand_nav_moe"
csv_path = f"{log_dir}/20260602_145556_655736_commands.csv"
csv_path = Path(csv_path)

save_fig = True
plot_fig = False
save_format = "svg"

if save_fig:
    save_dir = os.path.join(log_dir, csv_path.name.split(".")[0])
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(os.path.join(save_dir, "fps"), exist_ok=True)
    os.makedirs(os.path.join(save_dir, "segment"), exist_ok=True)
    os.makedirs(os.path.join(save_dir, "segment_cmd"), exist_ok=True)
    os.makedirs(os.path.join(save_dir, "segment_close"), exist_ok=True)
    os.makedirs(os.path.join(save_dir, "scatter"), exist_ok=True)

df = pd.read_csv(csv_path, header=None)
# print(df)

distance_threshold = 0.05  # m
heading_threshold = 3.0  # deg

In [ ]:
"fps analyze"
fps_list = []
with open(csv_path, "r") as f:
    reader = csv.reader(f)
    for row in reader:
        fps = float(row[1])
        fps_list.append(fps)

avg_fps = statistics.mean(fps_list)
std_fps = statistics.stdev(fps_list)

print(f"mean FPS: {avg_fps:.2f}")
print(f"std: {std_fps:.2f}")


# 过滤主峰
filtered_fps_list = [fps for fps in fps_list if not (95 <= fps <= 100)]
# 创建左右两个子图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1: Axes
ax2: Axes

# 左侧：原始数据的百分比直方图
weights_all = np.ones(len(fps_list)) / len(fps_list) * 100
ax1.hist(fps_list, bins=50, weights=weights_all, edgecolor="black")
ax1.set_xlabel("fps")
ax1.set_ylabel("percent (%)")
ax1.set_title("All data")

# 右侧：过滤主峰后的百分比直方图（分母仍为总样本数）
weights_filtered = np.ones(len(filtered_fps_list)) / len(fps_list) * 100
n, bins = np.histogram(filtered_fps_list, bins=50, weights=weights_filtered)
ax2.hist(fps_list, bins=50, weights=weights_all, edgecolor="black")
ax2.set_ylim([0, max(n)])
ax2.set_xlabel("fps")
ax2.set_ylabel("percent (%)")
ax2.set_title("Ignoring 95~100")

# 可选：设置相同的x轴范围，便于比较
x_min = min(min(fps_list), min(filtered_fps_list)) if filtered_fps_list else min(fps_list)
x_max = max(max(fps_list), max(filtered_fps_list)) if filtered_fps_list else max(fps_list)
ax1.set_xlim(x_min, x_max)
ax2.set_xlim(x_min, x_max)
ax1.grid(linestyle="--")
ax2.grid(linestyle="--")

plt.tight_layout()
if plot_fig:
    plt.show()
if save_fig:
    fig_name = f"fps_analyze.{save_format}"
    fig_path = os.path.join(save_dir, "fps", fig_name)
    fig.savefig(fig_path)
    print(f"saved: {fig_path}")
plt.close()

In [ ]:
"过滤没有走路的数据"
# 提取第8列为True的行号
true_rows = df[df[8] == True].index.tolist()
print(f"True总行数: {len(true_rows)}")
print(f"True行号示例: {true_rows[:10]}")

# 找出连续True的起始和结束点
segments = []
if len(true_rows) > 0:
    start = true_rows[0]
    end = true_rows[0]
    for i in range(1, len(true_rows)):
        if true_rows[i] == end + 1:  # 连续
            end = true_rows[i]
        else:  # 断开
            segments.append([start, end])
            start = true_rows[i]
            end = true_rows[i]
    segments.append([start, end])  # 最后一个段

print(f"连续True时间段: {segments}")
print(f"共{len(segments)}个连续时间段")

# TODO: 去除后四个数据全为0的片段
# 去除后四个数据全为0的片段
filtered_segments = []
for start, end in segments:
    segment_data = df.iloc[start:end + 1]
    last_4_cols = segment_data.iloc[:, -4:].values
    # 检查最后4列是否全为0
    if not (last_4_cols == 0).all():
        filtered_segments.append([start, end])

print(f"过滤后时间段: {filtered_segments}")
print(f"共{len(filtered_segments)}个时间段")
segments = filtered_segments

In [ ]:
def plot_segment(df, segments, seg_idx):
    """绘制指定时间段的数据

    Args:
        df: DataFrame
        segments: 连续True时间段列表
        seg_idx: 时间段索引(从0开始)
    """
    import matplotlib.pyplot as plt

    start, end = segments[seg_idx]
    segment_data = df.iloc[start:end + 1].copy()

    time_col = segment_data[0].values
    last_4_cols = segment_data.iloc[:, 9:9 + 5].values

    relative_time = time_col - time_col[0]
    fig, ax = plt.subplots(figsize=(14, 6))
    # labels = ["walk_vel_ref", "pos_x", "pos_y", "heading"]
    labels = ["walk_vel_ref", "pos_x", "pos_y", "rotate_vel_ref", "heading"]
    for i in range(len(labels)):
        ax.plot(relative_time, last_4_cols[:, i], label=labels[i])

    ax.set_xlabel('Relative Time (s)')
    ax.set_ylabel('Value')
    length = end - start + 1
    ax.set_title(f'Segment {seg_idx}: [{start}, {end}], start_time={time_col[0]:.3f}, length={length}')

    ax.legend()
    ax.grid(True)
    plt.tight_layout()

    if plot_fig:
        plt.show()
    plt.close()

    if save_fig:
        # 保存图片
        save_path = os.path.join(save_dir, "segment", f'{seg_idx}.{save_format}')
        fig.savefig(save_path)
        print(f"图片已保存到: {save_path}")


# 用for循环绘制所有时间段
for i in range(len(segments)):
    plot_segment(df, segments, i)
# plot_segment(df, segments, 0)

In [ ]:
final_points = []
for seg_idx in range(len(segments)):
    start, end = segments[seg_idx]
    segment_data = df.iloc[start:end + 1].copy()

    time_col = segment_data[0].values
    last_4_cols = segment_data.iloc[:, 9:9 + 5].values

    pos_x = last_4_cols[-1, 1]
    pos_y = last_4_cols[-1, 2]
    heading = last_4_cols[-1, 4]
    final_points.append([float(pos_x), float(pos_y), float(heading)])

print(final_points)
final_points = np.array(final_points)

In [ ]:
from matplotlib.patches import Circle

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1: Axes
ax2: Axes

# 左子图: pos_x vs pos_y 散点图
ax1.scatter(final_points[:, 0], final_points[:, 1])
ax1.grid(linestyle="--")
ax1.set_aspect('equal')
ax1.set_xlabel('x error (m)')
ax1.set_ylabel('y error (m)')
circle = Circle((0, 0), radius=distance_threshold, fill=False, edgecolor='red', linewidth=2)
ax1.add_patch(circle)
# 成功率
dist = np.sqrt(final_points[:, 0]**2 + final_points[:, 1]**2)
in_threshold = (dist <= distance_threshold).sum()
total = len(final_points)
ax1.set_title(f'success rate: {in_threshold}/{total} = {in_threshold/total*100:.1f}%')

# 右子图: heading 直方图 (单位: 度)
headings_deg = np.rad2deg(final_points[:, 2])
bins = np.arange(-10., 10., 1)  # -10 到 10 度, 每 1 度一个桶
ax2.hist(headings_deg, bins=bins, edgecolor='black')
ax2.axvline(x=-heading_threshold, color='red', linestyle='--', linewidth=1)
ax2.axvline(x=heading_threshold, color='red', linestyle='--', linewidth=1)
ax2.set_xlim(-10, 10)
# ax2.grid(linestyle="--")
ax2.set_xlabel('heading (deg)')
ax2.set_ylabel('count')
# 成功率
in_threshold_heading = (np.abs(headings_deg) <= heading_threshold).sum()
ax2.set_title(f'success rate: {in_threshold_heading}/{total} = {in_threshold_heading/total*100:.1f}%')

fig.suptitle("Termination state at navigation end", fontsize=14, fontweight='bold')
fig.tight_layout()
if save_fig:
    save_path = os.path.join(save_dir, "scatter", f'navigation_end.{save_format}')
    fig.savefig(save_path)
    print(f"图片已保存到: {save_path}")
if plot_fig:
    plt.show()
plt.close()

In [ ]:
# 绘制结束时间点之后的数据
post_segment_points = []  # 用于散点图: 每个segment结束后5秒的(pos_x, pos_y)
time_after = 2.0

for seg_idx in range(len(segments)):
    start, end = segments[seg_idx]
    end_time = df.iloc[end, 0]  # segment 结束时间点

    # 找到结束时间点之后的数据行
    post_data = df[(df[0] >= end_time) & (df[0] <= end_time + time_after)]

    if len(post_data) == 0:
        print(f"Segment {seg_idx}: 结束后5秒内无数据，跳过")
        continue

    time_col = post_data[0].values
    last_5_cols = post_data.iloc[:, 5:5 + 3].values  # pos_x, pos_y, heading

    relative_time = time_col - end_time

    fig, ax = plt.subplots(figsize=(14, 6))
    labels = ["pos_x", "pos_y", "heading"]
    for i in range(len(labels)):
        ax.plot(relative_time, last_5_cols[:, i], label=labels[i])

    ax.set_xlabel('Time after segment end (s)')
    ax.set_ylabel('Value')
    ax.set_title(f'Post-Segment {seg_idx}: [{start}, {end}], end_time={end_time:.3f}, post_data_len={len(post_data)}')
    ax.legend()
    ax.grid(True)
    plt.tight_layout()
    if plot_fig:
        plt.show()

    if save_fig:
        save_path = os.path.join(save_dir, "segment_close", f'{seg_idx}.{save_format}')
        fig.savefig(save_path)
        print(f"图片已保存到: {save_path}")
    plt.close()

    # 记录5秒后的终止状态（取最接近 end_time + 5 的那一行）
    target_time = end_time + time_after
    closest_idx = (post_data[0] - target_time).abs().idxmin()
    closest_row = df.iloc[closest_idx]
    pos_x = float(closest_row.iloc[5])  # pos_x 在列5
    pos_y = float(closest_row.iloc[6])  # pos_y 在列6
    heading = float(closest_row.iloc[7])  # heading 在列7
    post_segment_points.append([pos_x, pos_y, heading])

In [ ]:
from matplotlib.patches import Circle

# 终止状态散点图
post_segment_points = np.array(post_segment_points)
print(f"共 {len(post_segment_points)} 个segment有数据")
# print(post_segment_points)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1: Axes
ax2: Axes

# 左子图: pos_x vs pos_y 散点图
ax1.scatter(post_segment_points[:, 0], post_segment_points[:, 1])
ax1.grid(linestyle="--")
ax1.set_aspect('equal')
ax1.set_xlabel('x error (m)')
ax1.set_ylabel('y error (m)')
# 成功率
dist = np.sqrt(post_segment_points[:, 0]**2 + post_segment_points[:, 1]**2)
in_threshold = (dist <= distance_threshold).sum()
total = len(post_segment_points)
ax1.set_title(f'success rate: {in_threshold}/{total} = {in_threshold/total*100:.1f}%')

# 添加半径为 0.05 的圆
circle = Circle((0, 0), radius=distance_threshold, fill=False, edgecolor='red', linewidth=2)
ax1.add_patch(circle)

# 右子图: heading 直方图 (单位: 度)
headings_deg = np.rad2deg(post_segment_points[:, 2])
bins = np.arange(-10., 10., 1)  # -10 到 10 度, 每 1 度一个桶
ax2.hist(headings_deg, bins=bins, edgecolor='black')
ax2.axvline(x=-heading_threshold, color='red', linestyle='--', linewidth=1)
ax2.axvline(x=heading_threshold, color='red', linestyle='--', linewidth=1)
ax2.set_xlim(-10, 10)
# ax2.grid(linestyle="--")
ax2.set_xlabel('heading (deg)')
ax2.set_ylabel('count')
# 成功率
in_threshold_heading = (np.abs(headings_deg) <= heading_threshold).sum()
ax2.set_title(f'success rate: {in_threshold_heading}/{total} = {in_threshold_heading/total*100:.1f}%')

fig.suptitle("Termination state after close navigation", fontsize=14, fontweight='bold')
fig.tight_layout()
if plot_fig:
    plt.show()
if save_fig:
    save_path = os.path.join(save_dir, "scatter", f'navigation_close.{save_format}')
    fig.savefig(save_path)
    print(f"图片已保存到: {save_path}")
plt.close()

In [ ]:
def plot_segment2(df, segments, seg_idx):
    """绘制指定时间段的数据

    Args:
        df: DataFrame
        segments: 连续True时间段列表
        seg_idx: 时间段索引(从0开始)
    """
    import matplotlib.pyplot as plt

    start, end = segments[seg_idx]
    segment_data = df.iloc[start:end + 1].copy()

    time_col = segment_data[0].values
    last_4_cols = segment_data.iloc[:, 2:2 + 6].values

    relative_time = time_col - time_col[0]
    fig, ax = plt.subplots(figsize=(14, 6))
    labels = ["vx", "vy", "wz", "pos_x", "pos_y", "heading"]
    for i in range(len(labels)):
        ax.plot(relative_time, last_4_cols[:, i], label=labels[i])

    ax.set_xlabel('Relative Time (s)')
    ax.set_ylabel('Value')
    length = end - start + 1
    ax.set_title(f'Segment {seg_idx}: [{start}, {end}], start_time={time_col[0]:.3f}, length={length}')

    ax.legend()
    ax.grid(True)
    plt.tight_layout()

    if plot_fig:
        plt.show()
    plt.close()

    if save_fig:
        # 保存图片
        save_path = os.path.join(save_dir, "segment_cmd", f'{seg_idx}.{save_format}')
        fig.savefig(save_path)
        print(f"图片已保存到: {save_path}")



# # 用for循环绘制所有时间段
for i in range(len(segments)):
    plot_segment2(df, segments, i)
# plot_segment2(df, segments, 1)